In [1]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA, AirQuality
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR
from pytorch_lightning.loggers import TensorBoardLogger



def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [2]:
# dataset = MetrLA(root='./data/metrla')

# connectivity = dataset.get_connectivity(threshold=0.1,
#                                         include_self=False,
#                                         normalize_axis=1,
#                                         force_symmetric=False,
#                                         layout="edge_index")

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=connectivity,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)
# print(torch_dataset)

In [3]:
dataset = AirQuality(root='./data/aq36', impute_nans=True, small=True)

splitting = {"val_len": 0.1,
            "test_len": 0.2}


connectivity_sparse= {"method": "distance",
                    "threshold": 0.1,
                    "include_self": False,
                    "layout": "edge_index",
                     "normalize_axis":1}

adj = dataset.get_connectivity(**connectivity_sparse)

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=adj,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)

torch_dataset

SpatioTemporalDataset(n_samples=8736, n_nodes=36, n_channels=1)

In [4]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=6279}
{Validation dataloader: size=686}
{Test dataloader: size=1747}
{Predict dataloader: None}


In [5]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=5),
   'mae_step_4': torch_metrics.MaskedMAE(at=11)
}

model = models.DCRNNModel(input_size=1,exog_size=2, hidden_size = 64, output_size=1,
                          horizon=12, ff_size = 32, dropout = 0.1,
                          cache_support=True, n_layers = 2)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

In [6]:
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-3
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [7]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=30,
        mode='min'
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[0],
        gradient_clip_val=5,
       callbacks=[checkpoint_callback, early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        precision = '16-mixed',
        check_val_every_n_epoch = 5,
    logger=logger

    
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [8]:
trainer.fit(predictor, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | DCRNNModel       | 257 K  | train
-----------------------------------------------------------
257 K     Trainable params
0         Non-trainable params
257 K     Total params
1.029     Total estimated model params size (MB)
61        Modules in train mode
0         Modules in eval mode


Training: |                                                                | 0/? [00:00<?, ?it/s]

Only args ['edge_weight', 'x', 'edge_index', 'u'] are forwarded to the model (DCRNNModel).


Validation: |                                                              | 0/? [00:00<?, ?it/s]

Epoch 4, global step 490: 'val_mae' reached 36.15886 (best 36.15886), saving model to '/netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/AQI36/DCRNNModel/epoch=4-step=490.ckpt' as top 1


...

Epoch 144, global step 14210: 'val_mae' was not in top 1


Validation: |                                                                                                 …

Epoch 149, global step 14700: 'val_mae' was not in top 1


Validation: |                                                                                                 …

Epoch 154, global step 15190: 'val_mae' was not in top 1


In [9]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/AQI36/DCRNNModel/epoch=4-step=490.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/AQI36/DCRNNModel/epoch=4-step=490.ckpt


Testing: |                                                                                                    …

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     33.42961120605469     │
│         test_mae          │    31.964059829711914     │
│      test_mae_step_1      │    13.721465110778809     │
│      test_mae_step_2      │     22.9688663482666      │
│      test_mae_step_3      │     30.49463653564453     │
│      test_mae_step_4      │     35.27143478393555     │
│         test_mse          │       2593.7265625        │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 31.964059829711914,
  'test_mae_step_1': 13.721465110778809,
  'test_mae_step_2': 22.9688663482666,
  'test_mae_step_3': 30.49463653564453,
  'test_mae_step_4': 35.27143478393555,
  'test_mse': 2593.7265625,
  'test_loss': 33.42961120605469}]

<table>
  <tr>
    <th>Model</th>
    <th colspan="2" align="center">Metr LA</th>
    <th colspan="2" align="center">AirQuality 36</th>
      <th colspan="2" align="center">AirQuality Full</th>
  </tr>
  <tr>
    <th></th>
    <th>MAE</th>
    <th>MSE</th>
    <th>MAE</th>
    <th>MSE</th>
    <th>MAE</th>
    <th>MSE</th>
  </tr>
  <tr>
    <td>DCRNN Directed</td>
    <td>3.18</td>
    <td>39.67</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
  </tr>
    <tr>
    <td>DCRNN Undirected</td>
    <td>3.27</td>
    <td>42.04</td>
    <td>31.96</td>
    <td>2593.73</td>
    <td>21.21</td>
    <td>1414.68</td>
  </tr>
  <tr>
    <td>Graph Wavenet Directed</td>
    <td>3.16</td>
    <td>38.88</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
  </tr>
    <tr>
    <td>Graph Wavenet undirected</td>
    <td>3.24</td>
    <td>41.09</td>
    <td>30.63</td>
    <td>2344.78</td>
    <td>21.07</td>
    <td>1380.89</td>
  </tr>
  <tr>
    <td>Ours</td>
    <td>3.83</td>
    <td>59.78</td>
    <td>33.12</td>
    <td>2699.32</td>
    <td>23.12</td>
    <td>1558.18</td>
  </tr>
</table>